**SIGNATEのコンペ「【練習問題】銀行の顧客ターゲティング」**

ある銀行の顧客属性データおよび、過去のキャンペーンでの接触情報などから当該のキャンペーンの結果、口座を開設したかどうかを予測する。

[リンク](https://user.competition.signate.jp/ja/competition/detail/?competition=092375ab3c4a43c18c8277e1fd264aa9)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# train.csv, test.csv, submit_sample.csvの中を確認する

先に簡単そうなsubmit_sample.csv(提出用csvのサンプル)から

In [ ]:
# submit_sample.csv
submit_sample = pd.read_csv("submit_sample.csv",header=None, index_col=0)
submit_sample.columns = [""]
submit_sample.index.name = None
pd.concat([submit_sample.head(), submit_sample.tail()])

コンペのサイトでは各カラムの説明は以下のようににされている。

| カラム | ヘッダ名称 | データ型 | 説明                              |
|:-----------:|:------------|:---------|:---------------------------------|
| 0           | 無し        | int      | インデックスとして使用する通し番号 |
| 1           | 無し        | float    | 予測した定額預金申し込みの確率値   |

In [ ]:
submit_sample.describe().loc[["min", "max"]]

題意からも分かるが、maxとminを確認すると、0から1の範囲に収まる値で提出しなければならない。



次にtrain.csvとtest.csvを確認していく。

In [ ]:
# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None
pd.concat([train_data.head(), train_data.tail()])

In [ ]:
# test.csv
test_data = pd.read_csv("test.csv", index_col=0)
test_data.index.name = None
pd.concat([test_data.head(), test_data.tail()])

train.csvとtest.csvを比較すると、train.csvファイルは最終列に'y'列が追加されており、0か1かで値が入っている。
これは「定額預金申し込み有無（1:有り, 0:無し）」の値であることがコンペのサイトで説明されている。
また、各カラムの説明は以下のようににされている。

| カラム | ヘッダ名称 | データ型 | 説明 |
|:-----------:|:------------|:---------|:----------------------------------------------------------|
| 0   | `id`        | int      | 行の通し番号 |
| 1   | `age`       | int      | 年齢 |
| 2   | `job`       | varchar  | 職種 |
| 3   | `marital`   | varchar  | 未婚/既婚 |
| 4   | `education` | varchar  | 教育水準 |
| 5   | `default`   | varchar  | 債務不履行があるか（yes, no） |
| 6   | `balance`   | int      | 年間平均残高（€） |
| 7   | `housing`   | varchar  | 住宅ローン（yes, no） |
| 8   | `loan`      | varchar  | 個人ローン（yes, no） |
| 9   | `contact`   | varchar  | 連絡方法 |
| 10  | `day`       | int      | 最終接触日 |
| 11  | `month`     | char     | 最終接触月 |
| 12  | `duration`  | int      | 最終接触時間（秒） |
| 13  | `compaign`  | int      | 現キャンペーンにおける接触回数 |
| 14  | `pdays`     | int      | 経過日数（前キャンペーン接触後の日数） |
| 15  | `previous`  | int      | 接触実績（現キャンペーン以前の接触回数） |
| 16  | `poutcome`  | varchar  | 前回のキャンペーンの成果 |
| 17  | `y`         | boolean  | 定額預金申し込み有無（1:有り, 0:無し） ※train.csvのみ |

今回はidをindex_colに設定し、DataFrameを見やすくするためにヘッダ名称を削除

次に、使えそうなデータや使えなさそうなデータを探していく。
詳しくデータを確認する前にヘッダ名称から予想できることを羅列する。
1.   contactは要らないだろう

特に言うことも無し
2.   jobもまあ要らないだろう

一部の職が傾向的に強く出る可能性は否定できないが、ほとんどが関係なさそうで、説明変数として採用すると要らない変化を作りそう。

3.   housingとloanは似た傾向にある変数なので一つにする

housingのみ、あるいはloanのみ、あるいはどちらかでもyesであればyesとして処理する、のどれかが良いと思われる。
相関がなさそうならそもそも説明変数として採用しない。

4.   day,monthはyearの情報がないと使えないのでは

yearがあれば最終接触からどれだけの期間が空いたかのような変数にもできたが、dayとmanthだけだとそれも叶わず、使い道が無さそう。
そもそもpdaysで同じことができるので、それで良さそう。


他はデータを確認しながら考えていく。

In [ ]:
train_data.describe()

In [ ]:
train_data_varch = train_data[["job","marital","education","default","housing","loan","contact","month","poutcome","y"]]
for col in train_data_varch.columns:
    print(train_data_varch[col].value_counts())
    print()

これらのデータから各要素について

1.   age

使えるかもしれないが、高ければ、低ければの二極ではない数字ではありそうなので、木構造の解析では使えるかもといったところ

2.   job

unknownが少ないが、種別が多すぎるので、使えるのか少し怪しい

3.   marital

使える候補。相関を見る価値あり

4.   education

使える候補。unknownが少し多いかもしれない

5.   default

使える候補。yesなら申し込み無しなどの相関が期待できる

6.   balance

保留。housingやloanと似たような数値になるかもしれない

7.   housing

使える候補

8.   loan

使える候補

9.   contact

unknownが多すぎるために使えない

10.   day

使えない

11.   month

使えない

12.   duration

campaignのminが1なのにdurationのminが0なのが良く分からない。データとして残していないものがある?

13.   compaign

75%部分が3なのにmaxが64なのが気になる。外れ値を修正すれば使えそう

14.   pdays

ほとんどが-1(おそらく前キャンペーンで接触していない、前キャンペーンがyesでない)なので、-1かそうじゃないかの二極にすればいい数値になるかもしれない

15.   previous

これもほとんどが0なので、0かそうじゃないかの二極にすれば使えるかも

16.   poutcome

unknownが多すぎるが、pdaysが-1ならunknownになるのでは


In [ ]:
train_data["age"].plot.hist(bins=20, edgecolor="black")
plt.title("age Distribution")
plt.xlabel("age")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("ageとyの相関")
train_data["age"].corr(train_data["y"])

ageは中年層が多く、相関もかなり弱い。

In [ ]:
over60 = train_data[train_data["age"] >= 60]
print(over60["age"].corr(over60["y"]))
under25 = train_data[train_data["age"] <= 25]
print(under25["age"].corr(under25["y"]))

60歳以上、25歳以下に限定しても相関が弱い。

In [ ]:
job_y = pd.DataFrame({
  "job_cat": train_data["job"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("jobとyの相関")
job_y["job_cat"].corr(job_y["y"])

jobも相関が無さそうなので使わない。

In [ ]:
marital_y = pd.DataFrame({
  "marital_cat": train_data["marital"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("maritalとyの相関")
marital_y["marital_cat"].corr(marital_y["y"])

maritalも相関が弱い。

In [ ]:
education_y = pd.DataFrame({
  "education_cat": train_data["education"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("educationとyの相関")
education_y["education_cat"].corr(education_y["y"])

educationも相関が無さそう。

In [ ]:
default_y = pd.DataFrame({
  "default_cat": train_data["default"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("defaultとyの相関")
print(default_y["default_cat"].corr(default_y["y"]))
train_data[train_data["default"] == "yes"]["y"].value_counts()

defaultがyesのところだけ抜き出しても相関は無さそう。

In [ ]:
train_data["balance"].plot.hist(bins=30, edgecolor="black")
plt.title("balance Distribution")
plt.xlabel("balance")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("balanceとyの相関")
train_data["balance"].corr(train_data["y"])

もう少し調べてみる。

In [ ]:
balance_over14000 = train_data.loc[train_data["balance"] >= 14000, ["balance", "y"]]
print("balance(14000以上)とyの相関")
balance_over14000["balance"].corr(balance_over14000["y"])

In [ ]:
balance_under0 = train_data.loc[train_data["balance"] < 0, ["balance", "y"]]
print("balance(0未満)とyの相関")
balance_under0["balance"].corr(balance_under0["y"])

balanceも使えなさそう。

In [ ]:
housing_y = pd.DataFrame({
  "housing_cat": train_data["housing"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("housingとyの相関")
housing_y["housing_cat"].corr(housing_y["y"])

In [ ]:
loan_y = pd.DataFrame({
  "loan_cat": train_data["loan"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("housingとyの相関")
loan_y["loan_cat"].corr(loan_y["y"])

housingとloanは似た傾向にありそうだが、積極的に使えるほどの相関ではない。特にloan。

In [ ]:
contact_y = pd.DataFrame({
  "contact_cat": train_data["contact"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("loanとyの相関")
contact_y["contact_cat"].corr(contact_y["y"])

contactは意外と相関があったが、unknownが多い為、使いたくはない

In [ ]:
train_data["duration"].plot.hist(bins=30, edgecolor="black")
plt.title("duration Distribution")
plt.xlabel("duration")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("durationとyの相関")
train_data["duration"].corr(train_data["y"])

In [ ]:
train_data["duration"].value_counts().sort_index()

durationとyの相関はそこそこ強く出た。最低値が0なのが気になるがcountが2なので外れ値として処理しなくても良さそう。逆に高すぎる数字を出しているものは、一定値にまで抑えて使用したい。

mean 260.711295
std  260.091727

なので、一旦はおよそ3σにあたる1000に設定する。

In [ ]:
train_data["campaign"].plot.hist(bins=30, edgecolor="black")
plt.title("campaign Distribution")
plt.xlabel("campaign")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("campaignとyの相関")
train_data["campaign"].corr(train_data["y"])

In [ ]:
campaign_data = train_data["campaign"].value_counts().sort_index()
pd.concat([campaign_data.head(10), campaign_data.tail()])

campaignは意外と相関が無い。

In [ ]:
train_data["pdays"].value_counts().sort_index()

In [ ]:
train_data["pdays"].plot.hist(bins=30, edgecolor="black")
plt.title("pdays Distribution")
plt.xlabel("pdays")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("pdaysとyの相関")
train_data["pdays"].corr(train_data["y"])

In [ ]:
exclude_minus1 = train_data[train_data["pdays"] != -1]

exclude_minus1["pdays"].plot.hist(bins=30, edgecolor="black")
plt.title("pdays Distribution")
plt.xlabel("pdays")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("-1を除いたpdaysとyの相関")
exclude_minus1["pdays"].corr(exclude_minus1["y"])

-1の値が多すぎたためにそれを削除したデータでも相関を取ってみたが、pdaysもあまり強い相関があるとは言えなかった。

In [ ]:
previous_data = train_data["previous"].value_counts().sort_index()
pd.concat([previous_data.head(10), previous_data.tail()])

In [ ]:
train_data["previous"].plot.hist(bins=30, edgecolor="black")
plt.title("previous Distribution")
plt.xlabel("previous")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("previousとyの相関")
train_data["previous"].corr(train_data["y"])

In [ ]:
exclude_zero = train_data[train_data["previous"] != 0]

exclude_zero["previous"].plot.hist(bins=30, edgecolor="black")
plt.title("previous Distribution")
plt.xlabel("previous")
plt.ylabel("Count")
plt.grid(True)
plt.show()
print("0を除いたpreviousとyの相関")
exclude_zero["previous"].corr(exclude_zero["y"])

previousとyの相関も無い。

In [ ]:
poutcome_y = pd.DataFrame({
  "poutcome_cat": train_data["poutcome"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("poutcomeとyの相関")
poutcome_y["poutcome_cat"].corr(poutcome_y["y"])

In [ ]:
exclude_unknown = train_data[train_data["poutcome"] != "unknown"]

exclude_unknown_y = pd.DataFrame({
  "poutcome_cat": exclude_unknown["poutcome"].astype("category").cat.codes,
  "y": train_data["y"]
})
print("unknownを削除したpoutcomeとyの相関")
exclude_unknown_y["poutcome_cat"].corr(exclude_unknown_y["y"])

unkonnのデータを削除したputcomeはそこそこの相関を示した。ほかに使えそうな説明変数が少ないので、これをどうにか形にしたい

In [ ]:
unknown_data = train_data[train_data["poutcome"] == "unknown"]
pdays_previous_counts = unknown_data.groupby(["pdays", "previous"]).size().sort_index()
pdays_previous_counts

In [ ]:
known_data = train_data[train_data["poutcome"] != "unknown"]
pdays_previous_counts = known_data.groupby(["pdays", "previous"]).size().sort_index()
pdays_previous_counts

poutcomeの値がunknownだった場合、pdaysとpreviousの値もほぼ一定となる。

poutcomeカラムをone-hotベクトル化してみる。

In [ ]:
poutcome_y = train_data[["poutcome", "y"]]
poutcome_y = pd.get_dummies(poutcome_y, columns=["poutcome"])
poutcome_y.corr()

poutcomeがsuccessの値の時だけ相関が取れたので、これを採用する。

# 相関係数だけ見ると使えそうな変数が少なすぎる

現状で使えそうなものが
*   duration
*   poutcome(successの場合のみ)

だけという由々しき事態なので、もう少し探してみる。

In [ ]:
job_y = train_data[["job", "y"]]
job_y = pd.get_dummies(job_y, columns=["job"])
job_y.corr()

In [ ]:
marital_y = train_data[["marital", "y"]]
marital_y = pd.get_dummies(marital_y, columns=["marital"])
marital_y.corr()

In [ ]:
education_y = train_data[["education", "y"]]
education_y = pd.get_dummies(education_y, columns=["education"])
education_y.corr()

In [ ]:
contact_y = train_data[["contact", "y"]]
contact_y = pd.get_dummies(contact_y, columns=["contact"])
contact_y.corr()

見つからない！

仕方がないので一旦はdurationとpoutcomeだけでやってみる。

# 実装1(ロジスティック回帰)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

In [ ]:
def prepare_data1(df):

  # 使わないカラムを削除する
  prepared_df = df.drop(["age", "job", "marital", "education", "default",
                         "balance", "housing", "loan", "contact",
                         "day", "month", "campaign", "pdays", "previous"], axis=1)

  # poutcomeカラムからsuccessの値だけを取り出したpoutcome_successカラムを作成する
  prepared_df["poutcome_success"] = (prepared_df["poutcome"] == "success").astype(int)
  prepared_df = prepared_df.drop(["poutcome"], axis=1) # poutcomeカラムは削除する

  # durationが1000以上の値のデータを1000に統一する
  prepared_df.loc[prepared_df["duration"] >= 1000, "duration"] = 1000

  return prepared_df

In [ ]:
# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None

# データ成形
train_data = prepare_data1(train_data)
y = train_data["y"]
X = train_data.drop("y", axis = 1)

# 学習用データを学習用とテスト用に分けて実装
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_probs = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test) #閾値0.5

# 正解率（Accuracy）
print("Accuracy:", accuracy_score(y_test, y_pred))
# ROC-AUC（確率を出したときの評価指標）
print("ROC-AUC:", roc_auc_score(y_test, y_probs))
# 詳細なレポート
print(classification_report(y_test, y_pred))

1のrecallが低すぎる為、一度閾値を変更して実行。

In [ ]:
y_pred = (y_probs >= 0.2).astype(int) #閾値カスタム

# 正解率（Accuracy）
print("Accuracy:", accuracy_score(y_test, y_pred))
# ROC-AUC（確率を出したときの評価指標）
print("ROC-AUC:", roc_auc_score(y_test, y_probs))
# 詳細なレポート
print(classification_report(y_test, y_pred))

1のrecall自体は良化。ほかが微妙に悪化。

現在欲しいのは確率y_probsの値なので別の方向性で考えてみる。

In [ ]:
# キャリブレーション

base_model = LogisticRegression()
calibrated_model = CalibratedClassifierCV(base_model, cv=5, method='sigmoid')
# calibrated_model = CalibratedClassifierCV(base_model, cv=5, method='isotonic')

calibrated_model.fit(X_train, y_train)

y_probs_calibrated = calibrated_model.predict_proba(X_test)[:, 1]

y_probs = calibrated_model.predict_proba(X_test)[:, 1]     # 確率
y_pred = calibrated_model.predict(X_test)                  # 閾値0.5で分類

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_probs))
print(classification_report(y_test, y_pred))

brier_score = brier_score_loss(y_test, y_probs_calibrated)
print("Brier Score:", brier_score)

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_probs_calibrated, n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')  # 完全な理想線
plt.xlabel("Predicted Probability")
plt.ylabel("True Probability")
plt.title("Calibration Curve")
plt.grid()
plt.show()

視覚化した分には悪くなさそうだが、別に数値が良くなったわけではない。

# 実装2(LightGBM)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

In [ ]:
def prepare_data1(df):

  # 使わないカラムを削除する
  prepared_df = df.drop(["age", "job", "marital", "education", "default",
                         "balance", "housing", "loan", "contact",
                         "day", "month", "campaign", "pdays", "previous"], axis=1)

  #poutcomeカラムからsuccessの値だけを取り出したpoutcome_successカラムを作成する
  prepared_df["poutcome_success"] = (prepared_df["poutcome"] == "success").astype(int)
  prepared_df = prepared_df.drop(["poutcome"], axis=1) # poutcomeカラムは削除する

  # durationが1000以上の値のデータを1000に統一する
  prepared_df.loc[prepared_df["duration"] >= 1000, "duration"] = 1000

  return prepared_df

In [ ]:
# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None

# データ成形
train_data = prepare_data1(train_data)
y = train_data["y"]
X = train_data.drop("y", axis = 1)

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

# LightGBM 用データセット
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# パラメータ設定
params = {
  "objective": "binary",
  "metric": "auc",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 31,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 5,
  "seed": 42,
  "scale_pos_weight": len(y_train[y_train==0]) / len(y_train[y_train==1])
}

# モデル学習
model = lgb.train(
  params,
  train_data,
  valid_sets=[train_data, test_data],
  num_boost_round=1000,
  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)

# 確率予測
y_probs = model.predict(X_test)

# 閾値0.5で分類
y_pred = (y_probs >= 0.5).astype(int)

# 評価
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_probs))
print(classification_report(y_test, y_pred))

# がむしゃらにやってみる

以下、すべての説明変数を一旦採用し、減らしたり変形したりすることで点数の上下を観察して取捨選択する方法で行った。
現在の形になるまでの過程は省く。
モデルはLightGBM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import lightgbm as lgb

In [ ]:
def prepare_data2(df):
  prepared_df = df.copy()

  # pdays = -1 を「未接触フラグ」として別カラム化
  prepared_df["pdays_missing"] = (prepared_df["pdays"] == -1).astype(int)

  # educationをone-hotベクトル化
  prepared_df["education_tertiary"] = (prepared_df["education"] == "tertiary").astype(int)
  prepared_df["education_secondary"] = (prepared_df["education"] == "secondary").astype(int)
  prepared_df["education_primary"] = (prepared_df["education"] == "primary").astype(int)
  # prepared_df["education_unknown"] = (prepared_df["education"] == "unknown").astype(int)
  prepared_df.drop(["education"], axis=1, inplace=True)

  # poutcomeをone-hotベクトル化
  prepared_df["poutcome_success"] = (prepared_df["poutcome"] == "success").astype(int)
  # prepared_df["poutcome_failure"] = (prepared_df["poutcome"] == "failure").astype(int)
  # prepared_df["poutcome_other"] = (prepared_df["poutcome"] == "other").astype(int)
  prepared_df.drop(["poutcome"], axis=1, inplace=True)

  # durationが1000以上の値のデータを1000に統一する
  # prepared_df.loc[prepared_df["duration"] >= 1000, "duration"] = 1000
  # prepared_df.loc[prepared_df["previous"] >= 8, "previous"] = 8
  # prepared_df.loc[prepared_df["balance"] >= 10000, "balance"] = 10000
  # prepared_df.loc[prepared_df["campaign"] >= 12, "campaign"] = 12
  # prepared_df.loc[prepared_df["pdays"] >= 350, "pdays"] = 350

  # 数値変数のログ変換
  for col in ["balance", "campaign", "pdays", "duration", "previous"]:
    prepared_df[col] = prepared_df[col].apply(lambda x: np.log1p(x) if x > 0 else 0)
  # for col in ["balance", "campaign", "pdays"]:
  #   prepared_df[col] = prepared_df[col].apply(lambda x: x if x > 0 else 0)

  # カテゴリ変数をone-hotベクトル化
  # cat_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month"]
  cat_cols = ["marital", "housing", "loan", "contact", "month"]
  prepared_df = pd.get_dummies(prepared_df, columns=cat_cols, drop_first=True)
  prepared_df.drop(["job", "default"], axis=1, inplace=True)

  return prepared_df

In [ ]:
# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None

# データ成形
train_data = prepare_data2(train_data)
y = train_data["y"]
X = train_data.drop("y", axis = 1)

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

# LightGBM 用データセット
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# パラメータ設定
params = {
  "objective": "binary",
  "metric": "auc",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 31,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 5,
  "seed": 42,
  "scale_pos_weight": len(y_train[y_train==0]) / len(y_train[y_train==1])
}

# モデル学習
model = lgb.train(
  params,
  train_data,
  valid_sets=[train_data, test_data],
  num_boost_round=1000,
  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)

# 確率予測
y_probs = model.predict(X_test)

# 閾値0.5で分類
y_pred = (y_probs >= 0.5).astype(int)

# 評価
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_probs))
print(classification_report(y_test, y_pred))

lgb.plot_importance(model, importance_type="split", max_num_features=20)
lgb.plot_importance(model, importance_type="gain", max_num_features=20)

In [ ]:
# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None

# データ成形
train_data = prepare_data2(train_data)
y = train_data["y"]
X = train_data.drop("y", axis = 1)

# 学習用データを学習用とテスト用に分けて実装
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_probs = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test) #閾値0.5

# 正解率（Accuracy）
print("Accuracy:", accuracy_score(y_test, y_pred))
# ROC-AUC（確率を出したときの評価指標）
print("ROC-AUC:", roc_auc_score(y_test, y_probs))
# 詳細なレポート
print(classification_report(y_test, y_pred))

やってみて分かったこと

*   相関係数が高いものは役に立つが、相関係数が低いものが役に立たないわけじゃない
*   説明変数が少ない場合はロジスティック回帰、多い場合はLightGBMの方が優秀



# 結果出力

一旦prepared_data2で実装

In [ ]:
# 全データ使用

# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None

# test.csv
test_data = pd.read_csv("test.csv", index_col=0)
test_data.index.name = None

# データ成形
train_data = prepare_data2(train_data)
y_train = train_data["y"]
X_train = train_data.drop("y", axis = 1)
test_data = prepare_data2(test_data)

# LightGBM 用データセット
lgb_train = lgb.Dataset(X_train, label=y_train)

# パラメータ設定
params = {
  "objective": "binary",
  "metric": "auc",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 31,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 5,
  "seed": 42,
  "scale_pos_weight": len(y_train[y_train==0]) / len(y_train[y_train==1])
}

# モデル学習
model = lgb.train(
  params,
  lgb_train,
  num_boost_round=1000,
)

# 確率予測
y_probs = model.predict(test_data)

# 提出用csvファイル作成
submission = pd.DataFrame({
  "index": np.arange(1, len(y_probs)+1),
  "prediction": np.round(y_probs, 3)   # 小数点第3位まで
})

submission.to_csv("bankcustomer_submission1.csv",
                  index=False, header=False)


print("完了")

In [ ]:
submit_sample = pd.read_csv("bankcustomer_submission1.csv",header=None, index_col=0)
submit_sample.columns = [""]
submit_sample.index.name = None
pd.concat([submit_sample.head(), submit_sample.tail()])

In [ ]:
# refit at best_iteration

# train.csv
train_data = pd.read_csv("train.csv", index_col=0)
train_data.index.name = None

# test.csv
test_data = pd.read_csv("test.csv", index_col=0)
test_data.index.name = None

# データ成形
train_data = prepare_data2(train_data)
y_train = train_data["y"]
X_train = train_data.drop("y", axis=1)
test_data = prepare_data2(test_data)

# LightGBM 用データセット
lgb_train = lgb.Dataset(X_train, label=y_train)

# train/valid 分割
X_tr, X_val, y_tr, y_val = train_test_split(
  X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_valid = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# パラメータ設定
params = {
  "objective": "binary",
  "metric": "auc",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 31,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 5,
  "seed": 42,
  "scale_pos_weight": len(y_train[y_train==0]) / len(y_train[y_train==1])
}

# モデル学習（early stopping 有効）
model = lgb.train(
  params,
  lgb_train,
  valid_sets=[lgb_train, lgb_valid],
  valid_names=["train", "valid"],
  num_boost_round=1000,
  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)

# 確率予測
y_probs = model.predict(test_data)

# 提出用csvファイル作成
submission = pd.DataFrame({
  "index": np.arange(1, len(y_probs)+1),
  "prediction": np.round(y_probs, 3)   # 小数点第3位まで
})

submission.to_csv(
  "bankcustomer_submission2.csv",
  index=False, header=False
)

print("完了")


In [ ]:
submit_sample = pd.read_csv("bankcustomer_submission2.csv",header=None, index_col=0)
submit_sample.columns = [""]
submit_sample.index.name = None
pd.concat([submit_sample.head(), submit_sample.tail()])